# 🍽️ 템플릿 6 — AI 맛집 추천 서비스

상황/지역/예산을 입력하면 AI가 맛집을 추천. 데이터는 노트북에 내장 → 학생이 자유롭게 추가/수정.

## 핵심 구조
- **하드코딩된 맛집 DB** (학생이 확장)
- **Rule-based 필터** (지역/카테고리/가격대)
- **LLM 추천** (상황 맥락 보고 베스트 3개 선정)

## 학생이 가장 쉽게 손댈 곳
- `RESTAURANTS` 리스트: 본인 동네/학교 맛집 추가
- 필터 조건 추가 (혼밥 가능? 24시간? 분위기?)
- 카테고리 / 가격대 체계 변경

In [ ]:
!pip install -q gradio openai pandas

In [ ]:
SERVER_URL = "https://YOUR_URL.trycloudflare.com".strip().rstrip("/")
MODEL = "qwen2.5:7b-instruct"
assert "YOUR" not in SERVER_URL, "❌ SERVER_URL을 강사가 알려준 URL로 바꾸세요"

import httpx
from openai import OpenAI
client = OpenAI(base_url=f"{SERVER_URL}/v1", api_key="ollama",
                http_client=httpx.Client(headers={"User-Agent": "Mozilla/5.0"}))
print("✅ 준비 완료")

In [ ]:
import gradio as gr
import pandas as pd

# ===== 학생이 자유롭게 추가/수정할 맛집 DB =====
RESTAURANTS = pd.DataFrame([
    # 을지로/종로
    {"name": "을지로 OB베어", "category": "호프", "area": "을지로", "price": "₩",
     "specialty": "노가리 + 생맥주", "vibe": "레트로 노포, 시끌벅적"},
    {"name": "광장시장 박가네", "category": "분식", "area": "종로", "price": "₩",
     "specialty": "마약김밥, 빈대떡", "vibe": "전통시장, 정신없지만 활기참"},
    
    # 성수/뚝섬
    {"name": "성수 어니언", "category": "베이커리/카페", "area": "성수", "price": "₩₩",
     "specialty": "팡도르, 라떼", "vibe": "감성 카페, 인스타 핫플"},
    {"name": "성수 슈퍼판", "category": "이탈리안", "area": "성수", "price": "₩₩",
     "specialty": "수제 파스타", "vibe": "오픈 키친, 캐주얼"},
    
    # 강남
    {"name": "강남 미진", "category": "한식", "area": "강남", "price": "₩₩",
     "specialty": "메밀국수, 만두", "vibe": "조용한 노포, 혼밥 좋음"},
    {"name": "강남 본가설농탕", "category": "한식", "area": "강남", "price": "₩",
     "specialty": "설농탕, 깍두기", "vibe": "해장 맛집, 24시간"},
    
    # 홍대/연남
    {"name": "홍대 비스트로 마실", "category": "와인바", "area": "홍대", "price": "₩₩₩",
     "specialty": "내추럴 와인, 안주", "vibe": "데이트 분위기, 조명 어두움"},
    {"name": "연남 카페 어니언", "category": "카페", "area": "연남", "price": "₩₩",
     "specialty": "고소한 라떼", "vibe": "골목 감성"},
    
    # 신촌/이대
    {"name": "신촌 맛불금", "category": "고기집", "area": "신촌", "price": "₩₩",
     "specialty": "삼겹살, 김치찌개", "vibe": "친구들과 시끌벅적"},
    {"name": "이대 빨간잠수함", "category": "양식", "area": "이대", "price": "₩",
     "specialty": "오므라이스, 함박스테이크", "vibe": "학생 단골, 가성비"},
    
    # 이태원
    {"name": "이태원 박찬일", "category": "이탈리안", "area": "이태원", "price": "₩₩₩",
     "specialty": "트러플 파스타", "vibe": "기념일 분위기"},
    {"name": "이태원 부탄츄", "category": "에스닉", "area": "이태원", "price": "₩₩",
     "specialty": "쌀국수, 반미", "vibe": "베트남 본격 분위기"},
    
    # 여의도/마포
    {"name": "여의도 정육식당", "category": "고기집", "area": "여의도", "price": "₩₩₩",
     "specialty": "한우 등심", "vibe": "직장인 회식, 고급"},
    {"name": "마포 진대감", "category": "한식", "area": "마포", "price": "₩₩",
     "specialty": "갈비탕, 곰탕", "vibe": "노포, 어르신 단골"},
    
    # 압구정/청담
    {"name": "압구정 도산 카페밍크", "category": "카페", "area": "압구정", "price": "₩₩",
     "specialty": "소금빵, 콜드브루", "vibe": "감성 인테리어, 인스타용"},
    {"name": "청담 까사델비노", "category": "이탈리안", "area": "청담", "price": "₩₩₩",
     "specialty": "수제 라비올리", "vibe": "데이트, 분위기 끝판왕"},
    
    # ← 여기에 본인 동네 맛집 자유롭게 추가하세요!
    # {"name": "...", "category": "...", "area": "...", "price": "₩₩",
    #  "specialty": "...", "vibe": "..."},
])

# 카테고리/지역 옵션 자동 생성
AREAS = ["전체"] + sorted(RESTAURANTS["area"].unique().tolist())
CATEGORIES = ["전체"] + sorted(RESTAURANTS["category"].unique().tolist())
PRICE_MAP = {"₩": 1, "₩₩": 2, "₩₩₩": 3}

In [ ]:
def recommend(situation, area, category, max_price):
    if not situation.strip():
        return "💬 어떤 상황인지 입력해주세요 (왼쪽)"
    
    # 1단계: 룰베이스 필터
    df = RESTAURANTS.copy()
    if area != "전체":
        df = df[df["area"] == area]
    if category != "전체":
        df = df[df["category"] == category]
    df = df[df["price"].map(PRICE_MAP) <= PRICE_MAP[max_price]]
    
    if len(df) == 0:
        return "😢 조건에 맞는 맛집이 없어요. 필터를 완화해보세요."
    
    # 후보 텍스트
    candidates = df.to_dict("records")
    opts = "\n".join(
        f"{i+1}. **{r['name']}** ({r['category']}, {r['area']}, {r['price']})\n"
        f"   - 시그니처: {r['specialty']}\n"
        f"   - 분위기: {r['vibe']}"
        for i, r in enumerate(candidates[:12])
    )
    
    prompt = f"""당신은 친근한 맛집 큐레이터입니다. 사용자 상황에 가장 어울리는 맛집 3곳을 후보 중에서 골라 추천해주세요.

[사용자 상황]
{situation}

[후보 맛집]
{opts}

[추천 형식]
3곳을 골라서 다음 형식으로:

### 🥇 1순위: [이름]
- **추천 이유**: 1-2문장
- **이렇게 즐겨보세요**: 1문장 (예: "오후에 가서 창가 자리에 앉기")

### 🥈 2순위: ...
### 🥉 3순위: ..."""
    
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=700, temperature=0.7,
    )
    return resp.choices[0].message.content

with gr.Blocks(title="🍽️ AI 맛집 추천", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🍽️ AI 맛집 추천 서비스")
    gr.Markdown(f"현재 DB: **{len(RESTAURANTS)}개 맛집** (학생이 자유롭게 추가 가능)")
    
    with gr.Row():
        with gr.Column(scale=1):
            situation = gr.Textbox(
                label="🎬 상황 / 무엇을 원하는지 자세히",
                placeholder="예: 오랜만에 만나는 친구랑 분위기 좋은데서 가볍게 한 잔하고 싶어. 시끄러운 곳은 싫고.",
                lines=4,
            )
            area = gr.Dropdown(AREAS, value="전체", label="📍 지역")
            category = gr.Dropdown(CATEGORIES, value="전체", label="🍴 카테고리")
            price = gr.Radio(["₩", "₩₩", "₩₩₩"], value="₩₩₩",
                             label="💰 최대 가격대 (₩ 저렴 ~ ₩₩₩ 고급)")
            btn = gr.Button("➤ AI 추천받기", variant="primary", size="lg")
        with gr.Column(scale=2):
            result = gr.Markdown("👈 왼쪽에 상황 입력 후 추천받기 버튼을 눌러주세요")
    
    btn.click(recommend, [situation, area, category, price], result)

demo.launch(share=True)

---
## 🚀 바이브 코딩 확장 아이디어

### 쉬움
- 본인 학교/동네 맛집 50개로 늘리기
- 카테고리 추가 (분식, 베이커리, 디저트, 야식)
- 옵션 추가 (혼밥 가능?, 데이트?, 4인 이상?, 주차?)
- 점심/저녁/카페 시간대 추천

### 중간
- 맛집에 별점/리뷰 컬럼 추가
- 사용자 취향 학습 ("매운 거 좋아함" → 가중치)
- 추천 결과에 네이버 지도 링크 자동 생성
- 사진 컬럼 추가해서 결과에 같이 표시

### 도전적
- 학생들이 본인 추천 맛집 등록 (Google Sheets 연동)
- 친구 그룹별 합집합 추천 ("우리 셋이 다 좋아할 곳")
- 카카오/네이버 지도 API 연동 (좌표, 길찾기, 실시간 영업시간)
- 본인 위치 기반 가까운 순 정렬

### 🎁 자랑하기 팁
- "우리 학교 앞 맛집 30선" 컨셉으로 학교 커뮤니티에 공개
- 친구 청첩장 답례용 "결혼식 후 식사 추천" 같이 특정 상황 특화
- 동아리/소모임에 "오늘 어디서 회식?" 도구로 배포